In [ ]:
RUN_DATE = "2026-09-01"
SOURCE_FILE = "Sales.csv"

SOURCE_PATH = (
    f"Files/bronze/orders/landing/"
    f"ingestion_date={RUN_DATE}/{SOURCE_FILE}"
)

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(SOURCE_PATH)
)

print(raw_df.columns)
display(raw_df.limit(10))

from pyspark.sql import functions as F
#create bronze_df
COLUMN_MAPPING = {
    "Order Number": "order_number",
    "Line Item": "line_item",
    "Order Date": "order_date_raw",
    "Delivery Date": "delivery_date_raw",
    "CustomerKey": "customer_key",
    "StoreKey": "store_key",
    "ProductKey": "product_key",
    "Quantity": "quantity_raw",
    "Currency Code": "currency_code_raw",
}

bronze_df = raw_df.select(*[
    F.col(source_name).alias(target_name)
    for source_name, target_name in COLUMN_MAPPING.items()
])

bronze_df = (
    bronze_df
    .withColumn("_source_file", F.lit(SOURCE_FILE))
    .withColumn("_source_path", F.lit(SOURCE_PATH))
    .withColumn("_batch_id", F.lit("orders_initial_20260901_01"))
    .withColumn("_ingestion_date", F.to_date(F.lit(RUN_DATE)))
    .withColumn("_ingested_at_utc", F.current_timestamp())
)

#inspect
print(bronze_df.columns)
bronze_df.printSchema()
display(bronze_df.limit(10))

#
TARGET_TABLE = "bronze.orders_raw"
source_count = raw_df.count()

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Successfully wrote {TARGET_TABLE}")

#validate

target_df = spark.table(TARGET_TABLE)
target_count = target_df.count()

duplicate_key_groups = (
    target_df
    .groupBy("order_number", "line_item")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

metadata_null_rows = target_df.filter(
    F.col("_source_file").isNull()
    | F.col("_source_path").isNull()
    | F.col("_batch_id").isNull()
    | F.col("_ingestion_date").isNull()
    | F.col("_ingested_at_utc").isNull()
).count()

print(f"Source rows: {source_count:,}")
print(f"Target rows: {target_count:,}")
print(f"Duplicate-key groups: {duplicate_key_groups:,}")
print(f"Missing-metadata rows: {metadata_null_rows:,}")

assert target_count == source_count
assert duplicate_key_groups == 0
assert metadata_null_rows == 0

display(target_df.limit(10))